In [1]:
#@title Cell 1 - Mount Drive, configure, and safely attempt dependencies

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from typing import Any, Dict, List
from collections import defaultdict
import os
import sys
import re
import gc
import json
import time
import hashlib
import shutil
import subprocess
import platform

import numpy as np
import pandas as pd

OFFICIAL_REPOSITORY = (
    "https://github.com/RDXiaoLu/"
    "MambaCoral-DiffDet.git"
)

V3_TAG = (
    "multiseed_wbf_results_ready_v3_complete"
)

OUTPUT_TAG = (
    "multiseed_wbf_results_ready_v3_"
    "extension_final_combos"
)

PROTOCOL = "source_based"
SPLIT = "test"
IMAGE_SIZE = 640

# Leave blank until an official or user-supplied MCDD weight is available.
MCDD_WEIGHT_PATH = ""

# The official repository is Ultralytics-derived, not an MMDetection project.
# Enable only for a separate exploratory environment.
ATTEMPT_MMDET_STACK = False

ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/au-fish"),
    Path(
        "/content/drive/.shortcut-targets-by-id/"
        "1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish"
    ),
]

def resolve_project_root() -> Path:
    for candidate in ROOT_CANDIDATES:
        if (candidate / "dataset_v2").is_dir():
            return candidate.resolve()

    for root in [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/.shortcut-targets-by-id"),
    ]:
        if not root.exists():
            continue

        for dataset_root in root.rglob(
            "dataset_v2"
        ):
            if dataset_root.is_dir():
                return (
                    dataset_root.parent
                    .resolve()
                )

    raise FileNotFoundError(
        "Folder AU-FISH tidak ditemukan."
    )

PROJECT_ROOT = resolve_project_root()
DATASET_ROOT = PROJECT_ROOT / "dataset_v2"

V3_ROOT = (
    DATASET_ROOT
    / "experiments"
    / V3_TAG
)

V3_CANONICAL = (
    V3_ROOT
    / "canonical_data"
)

OUTPUT_ROOT = (
    DATASET_ROOT
    / "experiments"
    / OUTPUT_TAG
)

MCDD_OUTPUT_ROOT = (
    OUTPUT_ROOT
    / "mambacoral_attempt"
)

REPORTS_ROOT = (
    OUTPUT_ROOT
    / "reports"
)

MANIFESTS_ROOT = (
    OUTPUT_ROOT
    / "manifests"
)

for folder in [
    MCDD_OUTPUT_ROOT,
    REPORTS_ROOT,
    MANIFESTS_ROOT,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

ATTEMPT_LOG = (
    REPORTS_ROOT
    / "mambacoral_attempt_log.txt"
)

def append_log(message: str):
    with ATTEMPT_LOG.open(
        "a",
        encoding="utf-8",
    ) as handle:
        handle.write(
            str(message)
            + "\n"
        )

def run_logged(
    command: List[str],
    cwd: Path = None,
):
    append_log(
        "$ "
        + " ".join(
            command
        )
    )

    result = subprocess.run(
        command,
        cwd=(
            str(cwd)
            if cwd
            else None
        ),
        text=True,
        capture_output=True,
    )

    append_log(
        "RETURN CODE: "
        + str(
            result.returncode
        )
    )

    append_log(
        "STDOUT:\n"
        + result.stdout
    )

    append_log(
        "STDERR:\n"
        + result.stderr
    )

    return result

ATTEMPT_LOG.write_text(
    "",
    encoding="utf-8",
)

append_log(
    "MambaCoral-DiffDet "
    "experimental compatibility attempt"
)

append_log(
    "Python: "
    + platform.python_version()
)

base_install = run_logged([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "packaging",
    "ninja",
    "einops",
    "timm",
    "pyyaml",
    "pandas",
    "pycocotools",
])

mamba_install = run_logged([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-build-isolation",
    "causal-conv1d",
    "mamba-ssm",
])

if ATTEMPT_MMDET_STACK:
    mmdet_install = run_logged([
        sys.executable,
        "-m",
        "pip",
        "install",
        "openmim",
        "mmengine",
        "mmdet",
    ])
else:
    append_log(
        "MMDetection stack skipped: "
        "the official MCDD repository "
        "does not identify itself as "
        "an MMDetection implementation."
    )

dependency_status = {
    "python_version": (
        platform.python_version()
    ),
    "base_install_returncode": (
        base_install.returncode
    ),
    "mamba_install_returncode": (
        mamba_install.returncode
    ),
    "mamba_install_ok": (
        mamba_install.returncode
        == 0
    ),
    "official_environment_reference": (
        "Python 3.9, torch 2.2.2, "
        "torchvision 0.17.2, CUDA 12.1"
    ),
}

print(json.dumps(dependency_status, indent=2))

Mounted at /content/drive
{
  "python_version": "3.12.13",
  "base_install_returncode": 0,
  "mamba_install_returncode": 0,
  "mamba_install_ok": true,
  "official_environment_reference": "Python 3.9, torch 2.2.2, torchvision 0.17.2, CUDA 12.1"
}


In [2]:
#@title Cell 2 - Clone the official MambaCoral-DiffDet repository and inspect it

REPOSITORY_ROOT = Path(
    "/content/MambaCoral-DiffDet"
)

if REPOSITORY_ROOT.exists():
    shutil.rmtree(
        REPOSITORY_ROOT
    )

clone_result = run_logged([
    "git",
    "clone",
    "--depth",
    "1",
    OFFICIAL_REPOSITORY,
    str(
        REPOSITORY_ROOT
    ),
])

repository_status = {
    "official_repository": (
        OFFICIAL_REPOSITORY
    ),
    "clone_returncode": (
        clone_result.returncode
    ),
    "repository_root": str(
        REPOSITORY_ROOT
    ),
}

if clone_result.returncode == 0:
    commit_result = run_logged(
        [
            "git",
            "rev-parse",
            "HEAD",
        ],
        cwd=REPOSITORY_ROOT,
    )

    repository_status[
        "commit"
    ] = (
        commit_result.stdout.strip()
        if commit_result.returncode
        == 0
        else None
    )

    requirements_path = (
        REPOSITORY_ROOT
        / "requirements.txt"
    )

    if requirements_path.exists():
        repository_status[
            "requirements_path"
        ] = str(
            requirements_path
        )

        requirements_install = run_logged(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-r",
                str(
                    requirements_path
                ),
            ],
            cwd=REPOSITORY_ROOT,
        )

        repository_status[
            "requirements_install_returncode"
        ] = (
            requirements_install.returncode
        )

    repository_status[
        "candidate_scripts"
    ] = [
        str(
            path.relative_to(
                REPOSITORY_ROOT
            )
        )
        for path in REPOSITORY_ROOT.rglob(
            "*.py"
        )
        if path.name in {
            "detect.py",
            "val.py",
            "distill.py",
            "train.py",
        }
    ]

    repository_status[
        "bundled_weights"
    ] = [
        str(
            path.relative_to(
                REPOSITORY_ROOT
            )
        )
        for path in REPOSITORY_ROOT.rglob(
            "*.pt"
        )
    ]

else:
    repository_status[
        "status"
    ] = (
        "DEFERRED_REPOSITORY_CLONE_FAILED"
    )

with (
    MANIFESTS_ROOT
    / "mambacoral_repository_status.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        repository_status,
        handle,
        indent=2,
        ensure_ascii=False,
    )

print(json.dumps(repository_status, indent=2))

{
  "official_repository": "https://github.com/RDXiaoLu/MambaCoral-DiffDet.git",
  "clone_returncode": 0,
  "repository_root": "/content/MambaCoral-DiffDet",
  "commit": "47117f9f0a6e4798f1c1ce8dc00575defdd4a034",
  "requirements_path": "/content/MambaCoral-DiffDet/requirements.txt",
  "requirements_install_returncode": 1,
  "candidate_scripts": [
    "lib/distill.py",
    "lib/train.py",
    "lib/val.py",
    "lib/ultralytics/models/rtdetr/train.py",
    "lib/ultralytics/models/rtdetr/val.py",
    "lib/ultralytics/models/nas/val.py",
    "lib/ultralytics/models/fastsam/val.py",
    "lib/ultralytics/models/yolo/world/train.py",
    "lib/ultralytics/models/yolo/pose/train.py",
    "lib/ultralytics/models/yolo/pose/val.py",
    "lib/ultralytics/models/yolo/obb/train.py",
    "lib/ultralytics/models/yolo/obb/val.py",
    "lib/ultralytics/models/yolo/classify/train.py",
    "lib/ultralytics/models/yolo/classify/val.py",
    "lib/ultralytics/models/yolo/segment/train.py",
    "lib/ultralyti

In [3]:
#@title Cell 3 - Convert AU-FISH coral test data to a one-class YOLO dataset

from PIL import Image
import yaml

LOCAL_DATA_ROOT = Path(
    "/content/mcdd_au_fish"
)

LOCAL_IMAGE_ROOT = (
    LOCAL_DATA_ROOT
    / "images"
    / SPLIT
)

LOCAL_LABEL_ROOT = (
    LOCAL_DATA_ROOT
    / "labels"
    / SPLIT
)

for folder in [
    LOCAL_IMAGE_ROOT,
    LOCAL_LABEL_ROOT,
]:
    shutil.rmtree(
        folder,
        ignore_errors=True,
    )

    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

def load_json(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)

def relative_file_name(
    file_name: str,
):
    parts = list(
        Path(
            str(file_name)
            .replace(
                "\\",
                "/",
            )
        ).parts
    )

    while (
        parts
        and parts[0].lower()
        in {
            "train",
            "val",
            "test",
            "images",
        }
    ):
        parts = parts[1:]

    return Path(*parts)

source_gt = (
    V3_CANONICAL
    / PROTOCOL
    / SPLIT
    / "annotations.json"
)

source_images_root = (
    DATASET_ROOT
    / PROTOCOL
    / SPLIT
    / "images"
)

data = load_json(
    source_gt
)

images = {
    int(image["id"]):
    image
    for image in data["images"]
}

annotations_by_image = defaultdict(
    list
)

for annotation in data[
    "annotations"
]:
    if int(
        annotation[
            "category_id"
        ]
    ) == 1:
        annotations_by_image[
            int(
                annotation[
                    "image_id"
                ]
            )
        ].append(
            annotation
        )

conversion_rows = []

for image_id, image in images.items():
    relative = relative_file_name(
        image[
            "file_name"
        ]
    )

    candidates = [
        source_images_root
        / relative,
        source_images_root
        / Path(
            image[
                "file_name"
            ]
        ).name,
    ]

    source_image = next(
        (
            path
            for path
            in candidates
            if path.exists()
        ),
        None,
    )

    if source_image is None:
        matches = list(
            source_images_root.rglob(
                Path(
                    image[
                        "file_name"
                    ]
                ).name
            )
        )

        if len(matches) == 1:
            source_image = matches[0]

    if source_image is None:
        raise FileNotFoundError(
            image[
                "file_name"
            ]
        )

    destination_image = (
        LOCAL_IMAGE_ROOT
        / relative
    )

    destination_image.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        source_image,
        destination_image,
    )

    image_width = float(
        image["width"]
    )

    image_height = float(
        image["height"]
    )

    label_path = (
        LOCAL_LABEL_ROOT
        / relative
    ).with_suffix(
        ".txt"
    )

    label_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    lines = []

    for annotation in (
        annotations_by_image.get(
            image_id,
            [],
        )
    ):
        x, y, width, height = [
            float(value)
            for value
            in annotation[
                "bbox"
            ]
        ]

        center_x = (
            x + width / 2.0
        ) / image_width

        center_y = (
            y + height / 2.0
        ) / image_height

        lines.append(
            (
                f"0 "
                f"{center_x:.10f} "
                f"{center_y:.10f} "
                f"{width / image_width:.10f} "
                f"{height / image_height:.10f}"
            )
        )

    label_path.write_text(
        "\n".join(lines),
        encoding="utf-8",
    )

    conversion_rows.append({
        "image_id": image_id,
        "source_image": str(
            source_image
        ),
        "local_image": str(
            destination_image
        ),
        "coral_boxes": len(
            lines
        ),
    })

dataset_yaml = (
    LOCAL_DATA_ROOT
    / "mcdd_au_fish.yaml"
)

with dataset_yaml.open(
    "w",
    encoding="utf-8",
) as handle:
    yaml.safe_dump(
        {
            "path": str(
                LOCAL_DATA_ROOT
            ),
            "train": (
                f"images/{SPLIT}"
            ),
            "val": (
                f"images/{SPLIT}"
            ),
            "test": (
                f"images/{SPLIT}"
            ),
            "names": {
                0: "coral",
            },
        },
        handle,
        sort_keys=False,
    )

conversion_df = pd.DataFrame(
    conversion_rows
)

conversion_df.to_csv(
    MANIFESTS_ROOT
    / "mambacoral_aufish_conversion.csv",
    index=False,
)

append_log(
    "Converted AU-FISH images: "
    + str(
        len(
            conversion_df
        )
    )
)

append_log(
    "Dataset YAML: "
    + str(
        dataset_yaml
    )
)

display(conversion_df.head())
print("Dataset YAML:", dataset_yaml)

,image_id,source_image,local_image,coral_boxes
0,1,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/mcdd_au_fish/images/test/ikan kakatua...,0
1,2,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/mcdd_au_fish/images/test/ikan kakatua...,0
2,3,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/mcdd_au_fish/images/test/ikan kakatua...,0
3,4,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/mcdd_au_fish/images/test/ikan kakatua...,0
4,5,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/mcdd_au_fish/images/test/ikan kakatua...,0


Dataset YAML: /content/mcdd_au_fish/mcdd_au_fish.yaml


In [4]:
#@title Cell 4 - Safe inference and evaluation attempt

attempt_rows = []

published_reference = {
    "source": (
        "official MambaCoral-DiffDet "
        "repository table"
    ),
    "dataset": "SCoralDet",
    "map50": 0.843,
    "map50_95": 0.566,
    "parameters_million": 6.5,
    "gflops": 13.6,
    "reference_only": True,
}

if clone_result.returncode != 0:
    status = (
        "DEFERRED_REPOSITORY_CLONE_FAILED"
    )

elif mamba_install.returncode != 0:
    status = (
        "DEFERRED_MAMBA_KERNEL_INSTALL_FAILED"
    )

else:
    status = "ENVIRONMENT_READY"

candidate_weights = []

if MCDD_WEIGHT_PATH:
    candidate_weights.append(
        Path(
            MCDD_WEIGHT_PATH
        )
    )

if REPOSITORY_ROOT.exists():
    candidate_weights.extend(
        sorted(
            REPOSITORY_ROOT.rglob(
                "*.pt"
            )
        )
    )

weight_path = next(
    (
        path
        for path
        in candidate_weights
        if path.exists()
    ),
    None,
)

val_script = (
    REPOSITORY_ROOT
    / "val.py"
)

if status == "ENVIRONMENT_READY":
    if weight_path is None:
        status = (
            "DEFERRED_NO_MCDD_WEIGHT"
        )

        append_log(
            "No MCDD weight was found. "
            "Set MCDD_WEIGHT_PATH to an "
            "official downloaded checkpoint."
        )

    elif not val_script.exists():
        status = (
            "DEFERRED_VAL_SCRIPT_NOT_FOUND"
        )

    else:
        help_result = run_logged(
            [
                sys.executable,
                str(
                    val_script
                ),
                "--help",
            ],
            cwd=REPOSITORY_ROOT,
        )

        evaluation_command = [
            sys.executable,
            str(
                val_script
            ),
            "--weights",
            str(
                weight_path
            ),
            "--data",
            str(
                dataset_yaml
            ),
            "--imgsz",
            str(
                IMAGE_SIZE
            ),
            "--batch",
            "1",
            "--device",
            "0",
        ]

        evaluation_result = run_logged(
            evaluation_command,
            cwd=REPOSITORY_ROOT,
        )

        status = (
            "EVALUATION_COMMAND_COMPLETED"
            if evaluation_result.returncode
            == 0
            else "DEFERRED_EVALUATION_COMMAND_FAILED"
        )

        combined_output = (
            evaluation_result.stdout
            + "\n"
            + evaluation_result.stderr
        )

        map50_match = re.search(
            r"mAP(?:50|@0\\.5)[^0-9]*"
            r"([0-9]*\\.?[0-9]+)",
            combined_output,
            flags=re.IGNORECASE,
        )

        map5095_match = re.search(
            r"mAP(?:50[-_: ]?95|@0\\.5:0\\.95)"
            r"[^0-9]*([0-9]*\\.?[0-9]+)",
            combined_output,
            flags=re.IGNORECASE,
        )

        attempt_rows.append({
            "status": status,
            "weight_path": str(
                weight_path
            ),
            "val_script": str(
                val_script
            ),
            "command_returncode": (
                evaluation_result.returncode
            ),
            "parsed_map50": (
                float(
                    map50_match.group(1)
                )
                if map50_match
                else float("nan")
            ),
            "parsed_map50_95": (
                float(
                    map5095_match.group(1)
                )
                if map5095_match
                else float("nan")
            ),
            "metrics_parsed_from_stdout": (
                bool(
                    map50_match
                    or map5095_match
                )
            ),
        })

if not attempt_rows:
    attempt_rows.append({
        "status": status,
        "weight_path": (
            str(weight_path)
            if weight_path
            else None
        ),
        "val_script": (
            str(val_script)
            if val_script.exists()
            else None
        ),
        "parsed_map50": float("nan"),
        "parsed_map50_95": float("nan"),
        "metrics_parsed_from_stdout": (
            False
        ),
    })

attempt_df = pd.DataFrame(
    attempt_rows
)

attempt_df[
    "published_scoraldet_map50"
] = published_reference[
    "map50"
]

attempt_df[
    "published_scoraldet_map50_95"
] = published_reference[
    "map50_95"
]

attempt_df[
    "published_values_are_reference_only"
] = True

attempt_df.to_csv(
    REPORTS_ROOT
    / "mambacoral_attempt_summary.csv",
    index=False,
)

with (
    MANIFESTS_ROOT
    / "mambacoral_attempt_status.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        {
            "status": status,
            "official_repository": (
                OFFICIAL_REPOSITORY
            ),
            "dependency_status": (
                dependency_status
            ),
            "repository_status": (
                repository_status
            ),
            "published_reference": (
                published_reference
            ),
            "attempt_log": str(
                ATTEMPT_LOG
            ),
            "future_work_text": (
                "The official MCDD environment "
                "requires a dedicated compatible "
                "runtime and an official model "
                "checkpoint. Any published metric "
                "is reported only as literature "
                "context, not as an AU-FISH result."
            ),
        },
        handle,
        indent=2,
        ensure_ascii=False,
    )

append_log(
    "FINAL STATUS: "
    + status
)

display(attempt_df)
print("Attempt log:", ATTEMPT_LOG)
print("Final status:", status)

,status,weight_path,val_script,parsed_map50,parsed_map50_95,metrics_parsed_from_stdout,published_scoraldet_map50,published_scoraldet_map50_95,published_values_are_reference_only
0,DEFERRED_NO_MCDD_WEIGHT,None,None,NaN,NaN,False,0.843,0.566,True


Attempt log: /content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/experiments/multiseed_wbf_results_ready_v3_extension_final_combos/reports/mambacoral_attempt_log.txt
Final status: DEFERRED_NO_MCDD_WEIGHT
